# Machine Learning Notes
## Day 28: Column Transformer — Practice Notebook (Questions Only)

> **Watermark:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** ColumnTransformer in Scikit-learn  
> **Difficulty:** Beginner to Intermediate  

---
### Instructions:
Attempt each exercise in the empty code cell provided. Answers are NOT included in this notebook —
check the separate **Day28_Column_Transformer_Answers.ipynb** file once you're done.

### What You Will Practice:
1. Identifying which transformation each column needs
2. Basic ColumnTransformer with multiple transformers
3. Using the remainder parameter
4. Nesting a Pipeline inside ColumnTransformer (impute + scale)
5. ColumnTransformer combined with a model in a full Pipeline
6. Selecting columns automatically with make_column_selector
7. Mini end-to-end project — mixed dataset preprocessing

---

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

print('All libraries imported successfully!')
print('Notebook by: Amol Jagtap | amoljagtap3001@gmail.com')

---
## Section 1: Identifying Which Transformation Each Column Needs

**Concept Recap:**
- Numerical columns -> scaling (StandardScaler / MinMaxScaler)
- Nominal categorical -> OneHotEncoder
- Ordinal categorical -> OrdinalEncoder
- Numerical with missing values -> SimpleImputer first, then scaling

In [ ]:
# =============================================================
# EXERCISE 1: Classify each column and decide its transformation
# =============================================================

df = pd.DataFrame({
    'Age':          [25, 40, 33, 55, 29],
    'Salary':       [50000, 85000, np.nan, 120000, 60000],
    'City':         ['Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Delhi'],
    'Education':    ['Graduate', 'School', 'Postgraduate', 'Graduate', 'School'],
    'Approved':     ['Yes', 'No', 'No', 'Yes', 'No']
})
print('Dataset:')
print(df)

# TODO:
# 1. For each input column (Age, Salary, City, Education), decide the
#    correct transformation technique and print it as a dictionary
# 2. Note that 'Approved' is the TARGET column — explain in a comment
#    why it should NOT go through ColumnTransformer with the features

# Your code here:


---
## Section 2: Basic ColumnTransformer with Multiple Transformers

**Concept Recap:**
- `ColumnTransformer([('name', transformer, [columns]), ...])`
- Each tuple defines one transformation applied to a specific set of columns

In [ ]:
# =============================================================
# EXERCISE 2: Build a ColumnTransformer for Age, City, Education
# =============================================================

df2 = pd.DataFrame({
    'Age':        [25, 40, 33, 55, 29, 47],
    'City':       ['Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Delhi', 'Pune'],
    'Education':  ['Graduate', 'School', 'Postgraduate',
                    'Graduate', 'School', 'Postgraduate']
})

# TODO:
# 1. Create a ColumnTransformer that:
#    - Applies StandardScaler to 'Age'
#    - Applies OneHotEncoder to 'City'
#    - Applies OrdinalEncoder (order: School < Graduate < Postgraduate) to 'Education'
# 2. Fit and transform df2
# 3. Print the transformed array and the generated feature names using
#    get_feature_names_out()

# Your code here:


---
## Section 3: Using the remainder Parameter

**Concept Recap:**
- `remainder='drop'` (default) removes any column not explicitly listed
- `remainder='passthrough'` keeps unlisted columns unchanged, appended at the end

In [ ]:
# =============================================================
# EXERCISE 3: Compare remainder='drop' vs remainder='passthrough'
# =============================================================

df3 = pd.DataFrame({
    'Age':     [25, 40, 33, 55],
    'City':    ['Mumbai', 'Delhi', 'Pune', 'Mumbai'],
    'Salary':  [50000, 85000, 60000, 120000],   # not mentioned in transformer
    'Score':   [7.5, 8.2, 6.9, 9.1]              # not mentioned in transformer
})

# TODO:
# 1. Build a ColumnTransformer that only encodes 'City' with OneHotEncoder
#    (do not mention Age, Salary, or Score)
# 2. Fit and transform with the DEFAULT remainder ('drop') — observe that
#    Age, Salary, Score disappear from the output
# 3. Build a second ColumnTransformer identical to the first but with
#    remainder='passthrough' — observe that Age, Salary, Score are kept
# 4. Print both outputs and compare their shapes

# Your code here:


---
## Section 4: Nesting a Pipeline Inside ColumnTransformer

**Concept Recap:**
- A column with missing values AND a need for scaling requires TWO steps
- Wrap them in a small Pipeline (impute -> scale) and use that Pipeline
  as the transformer for that column inside ColumnTransformer

In [ ]:
# =============================================================
# EXERCISE 4: Handle a column with missing values + scaling together
# =============================================================

df4 = pd.DataFrame({
    'Age':     [25, 40, np.nan, 55, 29],
    'Fever':   [98.6, np.nan, 101.2, 99.5, np.nan],
    'Gender':  ['Male', 'Female', 'Female', 'Male', 'Female']
})
print('Dataset with missing values:')
print(df4)

# TODO:
# 1. Build a small Pipeline for 'Age': SimpleImputer(strategy='mean')
#    followed by StandardScaler
# 2. Build a small Pipeline for 'Fever': SimpleImputer(strategy='median')
#    followed by StandardScaler
# 3. Build a ColumnTransformer using these two pipelines for Age and Fever,
#    plus OneHotEncoder for Gender
# 4. Fit-transform df4 and print the final result (no NaN values should remain)

# Your code here:


---
## Section 5: ColumnTransformer + Model in a Full Pipeline

**Concept Recap:**
- Wrap ColumnTransformer as the first step of an outer Pipeline
- The model becomes the second step — one fit() call does everything

In [ ]:
# =============================================================
# EXERCISE 5: Full preprocessing + model pipeline
# =============================================================

np.random.seed(10)
n = 150
df5 = pd.DataFrame({
    'Age':     np.random.randint(20, 65, n),
    'Salary':  np.random.randint(25000, 150000, n),
    'City':    np.random.choice(['Mumbai', 'Delhi', 'Pune'], n),
})
# Simple synthetic target
df5['Buys'] = ((df5['Salary'] > 70000) & (df5['Age'] < 50)).astype(int)

# TODO:
# 1. Split df5 into X (Age, Salary, City) and y (Buys)
# 2. Train-test split (80/20)
# 3. Build a ColumnTransformer: StandardScaler on [Age, Salary],
#    OneHotEncoder on [City]
# 4. Build a full Pipeline: ColumnTransformer -> LogisticRegression
# 5. Fit the pipeline on training data and print test accuracy using
#    pipeline.score(X_test, y_test)

# Your code here:


---
## Section 6: Selecting Columns Automatically with make_column_selector

**Concept Recap:**
- `make_column_selector(dtype_include='number')` auto-selects numeric columns
- `make_column_selector(dtype_include='object')` auto-selects text/categorical columns
- Useful when you don't want to hard-code column names

In [ ]:
# =============================================================
# EXERCISE 6: Use make_column_selector instead of naming columns
# =============================================================

df6 = pd.DataFrame({
    'Age':       [22, 35, 58, 41, 29],
    'Income':    [25000, 60000, 120000, 85000, 40000],
    'City':      ['Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Delhi'],
    'Payment':   ['Card', 'UPI', 'Cash', 'UPI', 'Card']
})

# TODO:
# 1. Build a ColumnTransformer using make_column_selector to automatically
#    apply StandardScaler to ALL numeric columns
# 2. In the same ColumnTransformer, automatically apply OneHotEncoder to
#    ALL object (text/categorical) columns
# 3. Fit-transform df6 and print the result
# 4. Print which columns each selector picked up

# Your code here:


---
## Section 7: Mini End-to-End Project — Mixed Dataset Preprocessing

**Putting it all together!** Build a complete, leakage-free preprocessing
pipeline on a small medical-style dataset with numeric, missing, nominal,
and ordinal columns all at once.

In [ ]:
# =============================================================
# EXERCISE 7: Full mixed-type preprocessing pipeline
# =============================================================

raw = pd.DataFrame({
    'Age':           [25, 40, 33, 55, 29, 47, 38, 60],
    'Fever':         [98.6, 101.2, np.nan, 99.5, 100.1, np.nan, 98.9, 102.0],
    'Gender':        ['Male', 'Female', 'Female', 'Male',
                        'Female', 'Male', 'Female', 'Male'],
    'City':          ['Mumbai', 'Delhi', 'Pune', 'Mumbai',
                        'Delhi', 'Pune', 'Mumbai', 'Delhi'],
    'Cough_Severity':['Mild', 'Strong', 'Mild', 'Strong',
                        'Mild', 'Strong', 'Mild', 'Strong'],
    'Has_Disease':   ['No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes']
})
print('Raw Data:')
print(raw)

# TODO:
# Step 1: Split into X (all columns except Has_Disease) and y (Has_Disease)
# Step 2: Train-test split (75/25)
# Step 3: Build a ColumnTransformer with FOUR parts:
#         - 'Age': StandardScaler
#         - 'Fever': Pipeline(SimpleImputer(mean) -> StandardScaler)
#         - ['Gender', 'City']: OneHotEncoder(drop='first')
#         - 'Cough_Severity': OrdinalEncoder(categories=[['Mild','Strong']])
# Step 4: Fit on X_train ONLY, transform both X_train and X_test
# Step 5: Separately encode y with LabelEncoder (fit on y_train only)
# Step 6: Print the final transformed X_train, X_test, y_train, y_test
#         and the feature names from get_feature_names_out()

# Your code here:


---
## Summary — Concepts Covered

| Concept | Formula / Key Point |
|---|---|
| ColumnTransformer | Applies different transformers to different column subsets, then combines results |
| Syntax | `ColumnTransformer([('name', transformer, [columns]), ...])` |
| remainder='drop' | Default — unlisted columns are removed |
| remainder='passthrough' | Unlisted columns kept unchanged, appended at the end |
| Nested Pipeline | Use a Pipeline (impute -> scale) as a single transformer for one column |
| Full Pipeline | ColumnTransformer + model chained together — one fit() call |
| make_column_selector | Auto-selects columns by dtype instead of naming them manually |
| Fit rule | Always fit on training data only, transform on test data |

---
> **Notebook by:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Day 28 — Column Transformer  
> **Answers:** See Day28_Column_Transformer_Answers.ipynb